# 08 - Master Training Notebook (All Project Datasets)

This notebook uses **all datasets** in `ai-service/notebooks/dataset/`:
1. **Store Sales Classification (`train.csv` - 3.0M rows + `stores.csv` + `oil.csv` + `transactions.csv` + `holidays_events.csv`):**
   - 10-Epoch `HistGradientBoostingClassifier` with Accuracy/Loss plots, Pie Chart, Classification Report, and Confusion Matrix Heatmap
2. **Test Set Inference (`test.csv` - 28,512 rows):**
   - Predicts sales tiers on `test.csv` and visualizes predicted class distribution
3. **Store Sales Forecasting & Anomaly Detection (`train.csv` + `transactions.csv`):**
   - Uses `app.ml.train_forecast` and `app.ml.anomaly_detection` with Forecast & Anomaly charts
4. **Online Retail Forecasting & Anomaly Detection (`Online Retail.xlsx`):**
   - Trains revenue forecasting and order anomaly detection models on the e-commerce dataset
5. **Saves all trained models** to `ai-service/models/`

In [ ]:
import sys
from pathlib import Path
from uuid import uuid4

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils import shuffle

def find_ai_service_dir():
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / 'app').is_dir():
            return root
        if (root / 'ai-service' / 'app').is_dir():
            return root / 'ai-service'
    raise FileNotFoundError('Could not locate ai-service directory containing app/.')

AI_SERVICE_DIR = find_ai_service_dir()

if str(AI_SERVICE_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_DIR))

from app.ml.train_forecast import train_forecast_model
from app.ml.forecasting import predict_with_artifact
from app.ml.anomaly_detection import train_anomaly_model, detect_with_artifact

DATA_DIR = AI_SERVICE_DIR / 'notebooks' / 'dataset'
print('Imports complete! DATA_DIR:', DATA_DIR)

In [ ]:
# 1. Load ALL 3,000,888 rows of train.csv + stores + oil + transactions + holidays
raw_df = pd.read_csv(
    DATA_DIR / 'train.csv',
    dtype={'store_nbr': 'int16', 'family': 'str', 'sales': 'float32', 'onpromotion': 'int32'},
)
stores_df = pd.read_csv(DATA_DIR / 'stores.csv')
oil_df = pd.read_csv(DATA_DIR / 'oil.csv')
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].ffill().bfill()
tx_df = pd.read_csv(DATA_DIR / 'transactions.csv')
hol_df = pd.read_csv(DATA_DIR / 'holidays_events.csv')
hol_dates = set(hol_df.loc[hol_df['transferred'] == False, 'date'])

def enrich_store_sales(frame):
    frame = frame.merge(stores_df, on='store_nbr', how='left')
    frame = frame.merge(oil_df, on='date', how='left')
    frame = frame.merge(tx_df, on=['date', 'store_nbr'], how='left')
    frame['is_holiday'] = frame['date'].isin(hol_dates).astype('int8')
    frame['dcoilwtico'] = frame['dcoilwtico'].ffill().bfill().astype('float32')
    frame['transactions'] = frame['transactions'].fillna(0).astype('float32')
    dt = pd.to_datetime(frame['date'])
    frame['day_of_week'] = dt.dt.day_of_week.astype('int8')
    frame['month'] = dt.dt.month.astype('int8')
    frame['day'] = dt.dt.day.astype('int8')
    frame['year'] = dt.dt.year.astype('int16')
    return frame

raw_df = enrich_store_sales(raw_df)

labels = ['LOW_SALES', 'MEDIUM_SALES', 'HIGH_SALES', 'PEAK_SALES']
Y = pd.qcut(raw_df['sales'].rank(method='first'), q=4, labels=labels).astype(str).to_numpy()

cat_text_cols = ['family', 'city', 'state', 'type']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
raw_df[cat_text_cols] = encoder.fit_transform(raw_df[cat_text_cols].astype(str))

feature_cols = [
    'store_nbr', 'family', 'onpromotion', 'city', 'state', 'type',
    'cluster', 'dcoilwtico', 'transactions', 'is_holiday',
    'day_of_week', 'month', 'day', 'year',
]
X = raw_df[feature_cols].to_numpy(dtype=np.float32)
print(f'Loaded & merged ALL {len(X):,} rows from train.csv with {X.shape[1]} features!')

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
X, Y = shuffle(X, Y, random_state=101)
Xtrain, xtest, Ytrain, ytest = train_test_split(X, Y, test_size=0.2, random_state=101)
print('Xtrain:', Xtrain.shape, '| Ytrain:', Ytrain.shape, '| xtest:', xtest.shape, '| ytest:', ytest.shape)

In [ ]:
label_to_idx = {lbl: idx for idx, lbl in enumerate(labels)}
y_train_idx = np.array([label_to_idx[i] for i in Ytrain], dtype=np.int8)
y_test_idx = np.array([label_to_idx[i] for i in ytest], dtype=np.int8)

X_tr, X_val, y_tr, y_val = train_test_split(
    Xtrain, y_train_idx, test_size=0.2, random_state=101
)
print('Train split:', X_tr.shape, '| Validation split:', X_val.shape)

In [ ]:
# Train HistGradientBoostingClassifier over 10 epochs (25 iterations/epoch = 250 total)
categorical_idx = [0, 1, 3, 4, 5, 6, 9]
classifier_model = HistGradientBoostingClassifier(
    max_iter=25,
    max_leaf_nodes=127,
    learning_rate=0.12,
    categorical_features=categorical_idx,
    warm_start=True,
    random_state=101,
)
classes = np.arange(len(labels))
epochs = 10

history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
best_val_acc = -1.0

for epoch in range(1, epochs + 1):
    classifier_model.max_iter = epoch * 25
    classifier_model.fit(X_tr, y_tr)

    tr_prob = classifier_model.predict_proba(X_tr)
    val_prob = classifier_model.predict_proba(X_val)
    tr_acc = accuracy_score(y_tr, np.argmax(tr_prob, axis=1))
    val_acc = accuracy_score(y_val, np.argmax(val_prob, axis=1))
    tr_loss = log_loss(y_tr, tr_prob, labels=classes)
    val_loss = log_loss(y_val, val_prob, labels=classes)

    history['accuracy'].append(tr_acc)
    history['val_accuracy'].append(val_acc)
    history['loss'].append(tr_loss)
    history['val_loss'].append(val_loss)

    status = 'improved' if val_acc > best_val_acc else 'did not improve'
    if val_acc > best_val_acc:
        best_val_acc = val_acc

    print(f'Epoch {epoch}/{epochs} - accuracy: {tr_acc:.4f} - loss: {tr_loss:.4f} - val_accuracy: {val_acc:.4f} - val_loss: {val_loss:.4f} ({status})')

In [ ]:
pred = classifier_model.predict(xtest)
actual_label = y_test_idx
train_acc_pct = history['accuracy'][-1] * 100
val_acc_pct = history['val_accuracy'][-1] * 100
test_acc_pct = accuracy_score(actual_label, pred) * 100

print(f'Training Accuracy  : {train_acc_pct:.2f}%')
print(f'Validation Accuracy: {val_acc_pct:.2f}%')
print(f'Test Accuracy      : {test_acc_pct:.2f}%')
print('-' * 55)
print(classification_report(actual_label, pred, target_names=labels))

# Bar Chart of Training, Validation, and Test Accuracy
plt.figure(figsize=(7, 4))
acc_names = ['Training Accuracy', 'Validation Accuracy', 'Test Accuracy']
acc_values = [train_acc_pct, val_acc_pct, test_acc_pct]
bars = plt.bar(acc_names, acc_values, color=['#2563eb', '#10b981', '#f59e0b'], width=0.5)
for bar, val in zip(bars, acc_values):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 1.2, f'{val:.2f}%', ha='center', fontweight='bold')
plt.ylim(0, 105)
plt.title('Model Accuracy Summary (%)')
plt.ylabel('Accuracy (%)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
Cm = confusion_matrix(actual_label, pred)
plt.figure(figsize=(10, 7))
sns.heatmap(
    Cm,
    cmap='Blues',
    annot=True,
    fmt='d',
    annot_kws={'size': 10},
    xticklabels=labels,
    yticklabels=labels,
)
plt.title('Sales Tier Classification (All 3.0M Rows) - Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

In [ ]:
# Training Accuracy Plot
plt.figure(figsize=(10, 5))
plt.plot(history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title('Model Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Training Loss Plot (Separate)
plt.figure(figsize=(10, 4))
plt.plot(history['loss'], label='Training Loss', color='red', marker='o')
plt.title('Training Loss per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Log Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Validation Loss Plot (Separate)
plt.figure(figsize=(10, 4))
plt.plot(history['val_loss'], label='Validation Loss', color='orange', marker='o')
plt.title('Validation Loss per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Log Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Pie Charts using the SAME metric (Accuracy vs Error Rate) for Training and Test
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
ax1.pie(
    [train_acc_pct, 100 - train_acc_pct],
    labels=['Correct Predictions (Train Accuracy)', 'Train Error Rate'],
    colors=['#3b82f6', '#ef4444'],
    autopct='%1.2f%%',
    startangle=140,
)
ax1.set_title(f'Training Accuracy: {train_acc_pct:.2f}%')
ax2.pie(
    [test_acc_pct, 100 - test_acc_pct],
    labels=['Correct Predictions (Test Accuracy)', 'Test Error Rate'],
    colors=['#22c55e', '#ef4444'],
    autopct='%1.2f%%',
    startangle=140,
)
ax2.set_title(f'Test Accuracy: {test_acc_pct:.2f}%')
plt.tight_layout()
plt.show()

def accuracy_and_loss_Plot(hist):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(hist['accuracy'], label='Training Accuracy', marker='o')
    ax1.plot(hist['val_accuracy'], label='Validation Accuracy', marker='o')
    ax1.set_title('Training & Validation Accuracy')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(hist['loss'], label='Training Loss', color='red', marker='o')
    ax2.plot(hist['val_loss'], label='Validation Loss', color='orange', marker='o')
    ax2.set_title('Training & Validation Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

accuracy_and_loss_Plot(history)

### 2. Inference on `test.csv` (28,512 Future Rows)

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv', dtype={'store_nbr': 'int16', 'family': 'str', 'onpromotion': 'int32'})
test_enriched = enrich_store_sales(test_df.copy())
test_enriched[cat_text_cols] = encoder.transform(test_enriched[cat_text_cols].astype(str))
X_test_csv = test_enriched[feature_cols].to_numpy(dtype=np.float32)

test_preds = classifier_model.predict(X_test_csv)
test_df['predicted_tier'] = [labels[i] for i in test_preds]

tier_counts = test_df['predicted_tier'].value_counts().reindex(labels, fill_value=0)
plt.figure(figsize=(8, 4))
tier_counts.plot(kind='bar', color=['#94a3b8', '#60a5fa', '#3b82f6', '#1d4ed8'])
plt.title('Predicted Sales Tiers on test.csv (28,512 Rows)')
plt.ylabel('Row Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
test_df.head(10)

### 3. Store Sales Forecasting & Anomaly Detection (`train.csv` + `transactions.csv`)

In [ ]:
daily_df = raw_df.groupby('date', as_index=False)[['sales', 'onpromotion', 'transactions']].sum().sort_values('date').reset_index(drop=True)
store_rows = daily_df.to_dict(orient='records')
store_dataset_id = uuid4()

store_fc_artifact, store_fc_metrics, _ = train_forecast_model(
    dataset_id=store_dataset_id,
    date_column='date',
    value_column='sales',
    data_version='v1-store-full',
    rows=store_rows,
)
store_pred = predict_with_artifact(store_fc_artifact, periods=14)
obs_df = pd.DataFrame(store_pred['observations']).tail(60)
fc_df = pd.DataFrame(store_pred['forecast'])
obs_df['date'] = pd.to_datetime(obs_df['date'])
fc_df['date'] = pd.to_datetime(fc_df['date'])

plt.figure(figsize=(13, 5))
plt.plot(obs_df['date'], obs_df['value'], label='Historical Sales (Last 60 Days)', marker='o', markersize=3)
plt.plot(fc_df['date'], fc_df['value'], label='14-Day Forecast', linestyle='--', marker='o', markersize=5)
plt.title(f'Store Sales Forecast (MAE: {store_fc_metrics.mae:,.2f} | RMSE: {store_fc_metrics.rmse:,.2f})')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

store_anom_artifact, store_anom_metrics, _ = train_anomaly_model(
    dataset_id=store_dataset_id,
    feature_columns=['sales', 'onpromotion', 'transactions'],
    data_version='v1-store-full',
    rows=store_rows,
)
store_anom_res = detect_with_artifact(store_anom_artifact, store_rows)
flagged = set(item['rowIndex'] for item in store_anom_res['anomalies'])
daily_df['is_anomaly'] = daily_df.index.isin(flagged)

plt.figure(figsize=(10, 5))
plt.scatter(daily_df.loc[~daily_df['is_anomaly'], 'onpromotion'], daily_df.loc[~daily_df['is_anomaly'], 'sales'], alpha=0.5, label='Normal')
plt.scatter(daily_df.loc[daily_df['is_anomaly'], 'onpromotion'], daily_df.loc[daily_df['is_anomaly'], 'sales'], color='red', marker='X', s=80, label='Anomaly')
plt.title(f"Store Sales Anomaly Detection (Flagged Days: {store_anom_res['anomalyCount']})")
plt.xlabel('On Promotion')
plt.ylabel('Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4. Online Retail Forecasting & Anomaly Detection (`Online Retail.xlsx`)

In [ ]:
retail_df = pd.read_excel(DATA_DIR / 'Online Retail.xlsx')
retail_df = retail_df.dropna(subset=['InvoiceDate', 'Quantity', 'UnitPrice']).copy()
retail_df = retail_df[(retail_df['Quantity'] > 0) & (retail_df['UnitPrice'] > 0)].copy()
retail_df['Revenue'] = retail_df['Quantity'] * retail_df['UnitPrice']
retail_df['Date'] = pd.to_datetime(retail_df['InvoiceDate']).dt.strftime('%Y-%m-%d')

retail_daily = retail_df.groupby('Date', as_index=False)[['Revenue', 'Quantity']].sum().sort_values('Date').reset_index(drop=True)
retail_rows = retail_daily.to_dict(orient='records')
retail_dataset_id = uuid4()

retail_fc_artifact, retail_fc_metrics, _ = train_forecast_model(
    dataset_id=retail_dataset_id,
    date_column='Date',
    value_column='Revenue',
    data_version='v1-retail',
    rows=retail_rows,
)
retail_pred = predict_with_artifact(retail_fc_artifact, periods=14)
r_obs = pd.DataFrame(retail_pred['observations']).tail(60)
r_fc = pd.DataFrame(retail_pred['forecast'])
r_obs['date'] = pd.to_datetime(r_obs['date'])
r_fc['date'] = pd.to_datetime(r_fc['date'])

plt.figure(figsize=(13, 5))
plt.plot(r_obs['date'], r_obs['value'], label='Historical Revenue (Last 60 Days)', color='#10b981', marker='o', markersize=3)
plt.plot(r_fc['date'], r_fc['value'], label='14-Day Revenue Forecast', color='#f97316', linestyle='--', marker='o', markersize=5)
plt.title(f'Online Retail Revenue Forecast (MAE: {retail_fc_metrics.mae:,.2f} | RMSE: {retail_fc_metrics.rmse:,.2f})')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

retail_anom_artifact, retail_anom_metrics, _ = train_anomaly_model(
    dataset_id=retail_dataset_id,
    feature_columns=['Revenue', 'Quantity'],
    data_version='v1-retail',
    rows=retail_rows,
)
retail_anom_res = detect_with_artifact(retail_anom_artifact, retail_rows)
print(f"Online Retail rows: {len(retail_df):,} | Daily periods: {len(retail_daily)} | Anomalies flagged: {retail_anom_res['anomalyCount']}")

In [ ]:
# 5. Save ALL trained models locally
models_dir = AI_SERVICE_DIR / 'models'
models_dir.mkdir(exist_ok=True)

joblib.dump({'model': classifier_model, 'encoder': encoder, 'feature_cols': feature_cols, 'labels': labels}, models_dir / 'sales_tier_classifier.joblib')
joblib.dump(store_fc_artifact, models_dir / 'store_sales_forecast_model.joblib')
joblib.dump(store_anom_artifact, models_dir / 'store_sales_anomaly_model.joblib')
joblib.dump(retail_fc_artifact, models_dir / 'online_retail_forecast_model.joblib')
joblib.dump(retail_anom_artifact, models_dir / 'online_retail_anomaly_model.joblib')

print('All 5 models saved to:', models_dir)